<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c37-header.png" alt="Nextia Learning · Recommendation and Personalisation: Show Each Person What Matters" width="100%">

# Recommend for an online retailer

**[Recommend for an online retailer](https://learning.nextia-ai.com/courses/recommendations/m10/recommend-for-an-online-retailer/)** · Recommendation and Personalisation: Show Each Person What Matters · Module 10, case study 1 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will** compare popularity, co-occurrence, ALS and a two-stage ranker on two years of real invoices from a UK online shop, with a time split and customer segments, and decide what the shop should show and for whom.

| | |
|---|---|
| **Time** | About 2.5 hours with the lesson page; the notebook runs in about 3 minutes |
| **Needs** | An internet connection for the setup cells. No account. Not the course project. |
| **You should know** | Recall@10 and segments, from [Evaluate by segment](https://learning.nextia-ai.com/courses/recommendations/m03/evaluate-by-segment/). The two-stage ranker, from [A learned ranker](https://learning.nextia-ai.com/courses/recommendations/m07/a-learned-ranker/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1, SciPy 1.18.1, matplotlib 3.11.2, implicit 0.7.3 and LightGBM 4.7.0, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*). Colab and Kaggle were not run. |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/recommendations/m10/recommend-for-an-online-retailer/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C37/M10-L01-recommend-for-an-online-retailer/recommend-for-an-online-retailer-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: NumPy 2.5.4, pandas 3.1.1, SciPy 1.18.1, implicit 0.7.3 (ALS) and LightGBM 4.7.0 (the ranker), and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

On a Mac, LightGBM needs the OpenMP library from Homebrew. If `import lightgbm` fails with `libomp.dylib`, run `brew install libomp` in a terminal, then restart the kernel.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 scipy==1.18.1 matplotlib==3.11.2 implicit==0.7.3 lightgbm==4.7.0
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'pandas', 'scipy', 'implicit', 'lightgbm']:
    print(name, version(name))

## Setup: get the data

Run the next cell. It makes a folder `online-retailer-recs` with a `data` folder in it. It downloads `online_retail_ii.csv.gz` (15 MB) from the course's labs repository and checks the file's checksum (SHA-256). It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready:` and the file with its size. If you run the cell again, it uses the file that is already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `online-retailer-recs` (in Colab: the **Files** panel on the left), then run the cell again.

The data is **Online Retail II** by Daqing Chen, from the UCI Machine Learning Repository ([doi.org/10.24432/C5CG6D](https://doi.org/10.24432/C5CG6D)), licensed under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). The course joined the two Excel sheets of the source into one compressed CSV file and changed nothing else; the [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C37/M10-L01-recommend-for-an-online-retailer/dataset.md) lists the details. It is real data of a UK online shop. Customer IDs are numbers only; do not try to identify anyone.

In [ ]:
# Setup: download the data into online-retailer-recs/data/ and check it.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C05/case-studies/data/"
FILES = {
    "online_retail_ii.csv.gz": "c5ab45f73700a1411440c2cd6b7d85611d22cdc72b3594c2d29d37261e977e4f",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into another project. If the cell ran before, start from the parent.
if Path(".nextia-notebook").exists():
    os.chdir("..")
    HERE = Path.cwd()
WORK = Path("online-retailer-recs").resolve()
(WORK / "data").mkdir(parents=True, exist_ok=True)
(WORK / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = WORK / "data" / name
    if not (path.exists() and sha256(path) == expected):
        if (HERE / name).exists():  # a copy next to the notebook
            shutil.copy(HERE / name, path)
        else:
            download(LABS + name, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder online-retailer-recs and run again.")

os.chdir(WORK)
print("Ready:", ", ".join(f"data/{n} ({Path('data', n).stat().st_size / 1e6:.1f} MB)" for n in FILES))

## Setup: helpers

The next cell defines the case study's tools. Read the comments if you like; the lesson page explains each one.

- `read(path)` and `clean(raw)` load the invoice lines and clean them, and count what each step removed.
- `Period(lines, start, end)` is the time split: everything known before `start`, and what each customer bought from `start` to the day before `end`. It also gives each customer a segment (new, occasional or loyal).
- `personal(...)`, `top_k(...)` and `fill(...)` turn scores into lists of 10 products; customers with no history get a fallback list.
- `evaluate(period, lists, words)` measures the lists: recall@10 with a 95% interval, NDCG@10, coverage, novelty, diversity, recall on products new to the customer, and recall by segment. The same measures as the course project's evaluation harness.
- `expect(...)` and `expect_hash(...)` check your answers without showing them.

Run the cell. It prints nothing.

In [ ]:
# Helpers: cleaning, the time split, lists and measures (the same code as the lesson page).
import hashlib, json, time, warnings
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

NOT_PRODUCTS = ["POST", "DOT", "C2", "C3", "M", "D", "S", "B", "BANK CHARGES", "ADJUST", "ADJUST2", "AMAZONFEE",
                "CRUK", "TEST001", "TEST002", "GIFT"]


SEGMENTS = ("new", "occasional", "loyal")


K = 10


def read(path):
    """The raw invoice lines (1,067,371 rows). Invoice and StockCode are text."""
    return pd.read_csv(path, dtype={"Invoice": str, "StockCode": str}, parse_dates=["InvoiceDate"])


def clean(raw):
    """Clean invoice lines and a count of what each step removed."""
    steps = {"lines": len(raw)}
    d = raw[~raw.duplicated()]
    steps["repeated lines"] = len(raw) - len(d)
    steps["no customer ID"] = int(d["Customer ID"].isna().sum())
    d = d[d["Customer ID"].notna()]
    code = d["StockCode"].str.upper()
    not_product = code.isin(NOT_PRODUCTS) | code.str.startswith("GIFT_")
    steps["not a product"] = int(not_product.sum())
    d, code = d[~not_product], code[~not_product]
    free = d["Price"] <= 0
    steps["price 0 or less"] = int(free.sum())
    d, code = d[~free], code[~free]
    lines = pd.DataFrame({
        "invoice": d["Invoice"].to_numpy(),
        "date": d["InvoiceDate"].to_numpy(),
        "customer": d["Customer ID"].astype(int).astype(str).to_numpy(),
        "product": code.to_numpy(),
        "name": d["Description"].fillna("").str.strip().to_numpy(),
        "quantity": d["Quantity"].to_numpy(),
        "price": d["Price"].to_numpy(),
        "country": d["Country"].to_numpy(),
    })
    lines["returned"] = lines["invoice"].str.startswith("C")
    steps["kept"] = len(lines)
    steps["return lines"] = int(lines["returned"].sum())
    return lines, steps


def product_names(lines):
    """The most common name of each product code."""
    n = lines[lines["name"] != ""].groupby(["product", "name"]).size().reset_index(name="n")
    n = n.sort_values(["product", "n", "name"], ascending=[True, False, True]).drop_duplicates("product")
    return n.set_index("product")["name"]


def product_words(name):
    """The words of a product's name, for diversity: ALARM CLOCK BAKELIKE PINK -> {ALARM, CLOCK, BAKELIKE, PINK}."""
    return frozenset(w for w in str(name).upper().replace(",", " ").split() if w.isalpha())


def same_kind(a, b):
    """Two products are the same kind when their names share at least half of their words (Jaccard >= 0.5)."""
    return len(a | b) > 0 and len(a & b) / len(a | b) >= 0.5


class Period:
    """Everything known before `start`, and what customers bought from `start` to the day before `end`."""

    def __init__(self, lines, start, end):
        self.start, self.end = pd.Timestamp(start), pd.Timestamp(end)
        before = lines[lines["date"] < self.start]
        buys = before[~before["returned"]]
        net = before.groupby(["customer", "product"])["quantity"].sum()
        pairs = buys.groupby(["customer", "product"]).agg(invoices=("invoice", "nunique"), first=("date", "min"),
                                                          last=("date", "max"))
        keep = pairs.index.isin(net[net > 0].index)
        self.returned_pairs = int((~keep).sum())
        self.pairs = pairs[keep].reset_index()
        self.buys = buys
        self.products = np.array(sorted(buys["product"].unique()))
        self.pidx = {p: i for i, p in enumerate(self.products)}
        self.users = np.array(sorted(self.pairs["customer"].unique()))
        self.uidx = {u: i for i, u in enumerate(self.users)}
        rows = self.pairs["customer"].map(self.uidx).to_numpy()
        cols = self.pairs["product"].map(self.pidx).to_numpy()
        inv = self.pairs["invoices"].to_numpy().astype(np.float32)
        shape = (len(self.users), len(self.products))
        self.X = sp.csr_matrix((inv, (rows, cols)), shape=shape)          # invoices per (customer, product)
        self.B = sp.csr_matrix((np.ones_like(inv), (rows, cols)), shape=shape)  # 1 = bought (and kept)
        days = (self.start - self.pairs["last"]).dt.days.to_numpy()
        self.D = sp.csr_matrix((days.astype(np.float32) + 1, (rows, cols)), shape=shape)  # days since + 1
        self.item_customers = np.asarray(self.B.sum(0)).ravel()
        # Products on sale: sold in the 90 days before the period.
        recent = buys[buys["date"] >= self.start - pd.Timedelta(days=90)]
        self.on_sale = np.zeros(len(self.products), bool)
        self.on_sale[[self.pidx[p] for p in recent["product"].unique()]] = True
        # What customers bought in the period (minus products returned in full in the period).
        inside = lines[(lines["date"] >= self.start) & (lines["date"] < self.end)]
        net_in = inside.groupby(["customer", "product"])["quantity"].sum()
        got = inside[~inside["returned"]].groupby(["customer", "product"]).size().index
        got = got[got.isin(net_in[net_in > 0].index)]
        t = pd.DataFrame({"customer": got.get_level_values(0), "product": got.get_level_values(1)})
        self.truth = t.groupby("customer")["product"].agg(set).to_dict()
        self.customers = np.array(sorted(self.truth))
        invoices = buys.groupby("customer")["invoice"].nunique()
        n = invoices.reindex(self.customers).fillna(0).to_numpy()
        self.segment = np.where(n == 0, "new", np.where(n <= 2, "occasional", "loyal"))
        self.invoices_before = invoices
        self.last_day = buys.groupby("customer")["date"].max()

    def popularity(self, days=None):
        """Customers who bought each product (in the last `days` days, or ever)."""
        b = self.buys if days is None else self.buys[self.buys["date"] >= self.start - pd.Timedelta(days=days)]
        c = b.groupby("product")["customer"].nunique()
        return c.reindex(self.products).fillna(0).to_numpy()


def top_k(scores, k=K):
    """The k best columns of each row, best first."""
    k = min(k, scores.shape[1])
    part = np.argpartition(-scores, k - 1, axis=1)[:, :k]
    order = np.argsort(-np.take_along_axis(scores, part, axis=1), axis=1, kind="stable")
    return np.take_along_axis(part, order, axis=1)


def fill(lists, fallback, k=K):
    """Make every list k long: missing places (-1) take the fallback's products, in order, without repeats."""
    out = []
    for row in lists:
        row = [int(p) for p in row if p >= 0]
        seen = set(row)
        for p in fallback:
            if len(row) >= k:
                break
            if p not in seen:
                row.append(int(p))
                seen.add(int(p))
        out.append(row[:k])
    return np.array(out)


def known_rows(period):
    """For each test customer: their row in the matrices, or -1 for a new customer."""
    return np.array([period.uidx.get(c, -1) for c in period.customers])


def personal(period, scores_of, fallback, k=K):
    """Lists from a scoring function for known customers; the fallback for the others."""
    rows = known_rows(period)
    lists = np.full((len(rows), k), -1)
    known = rows >= 0
    if known.any():
        s = scores_of(rows[known]).astype(np.float64)
        top = top_k(s, k)
        good = np.take_along_axis(s, top, axis=1) > 0
        lists[known] = np.where(good, top, -1)
    return fill(lists, fallback, k)


def bootstrap(values, n=1000, seed=37):
    values = np.asarray(values, dtype=float)
    rng = np.random.default_rng(seed)
    means = values[rng.integers(0, len(values), (n, len(values)))].mean(axis=1)
    return float(values.mean()), float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))


def per_customer(period, lists, k=K):
    """recall@k and NDCG@k per customer, and recall on products new to the customer (NaN if none)."""
    disc = 1 / np.log2(np.arange(2, k + 2))
    rec, ndcg, disc_rec = [], [], []
    for i, c in enumerate(period.customers):
        truth = {period.pidx[x] for x in period.truth[c] if x in period.pidx}
        n = len(period.truth[c])  # products never sold before still count: no method can show them
        hits = np.array([1.0 if q in truth else 0.0 for q in lists[i][:k]])
        rec.append(hits.sum() / n)
        ndcg.append((hits * disc).sum() / disc[:min(n, k)].sum())
        r = period.uidx.get(c, -1)
        own = set(period.X[r].indices) if r >= 0 else set()
        new = truth - own
        n_new = len(period.truth[c]) - len(truth & own)
        disc_rec.append(len(set(lists[i][:k]) & new) / n_new if n_new else np.nan)
    return np.array(rec), np.array(ndcg), np.array(disc_rec)


def evaluate(period, lists, words, k=K):
    rec, ndcg, disc = per_customer(period, lists, k)
    shown = np.unique(lists[:, :k])
    users = len(period.users)
    info = -np.log2((period.item_customers + 1) / (users + 1))
    div = []
    for row in lists[:, :k]:
        pairs = [(a, b) for i, a in enumerate(row) for b in row[i + 1:]]
        div.append(sum(not same_kind(words[a], words[b]) for a, b in pairs) / len(pairs))
    out = {
        "recall": bootstrap(rec), "ndcg": float(ndcg.mean()),
        "coverage": float(period.on_sale[shown].sum() / period.on_sale.sum()),
        "novelty": float(info[lists[:, :k]].mean()), "diversity": float(np.mean(div)),
        "discovery": float(np.nanmean(disc)),
        "segments": {s: bootstrap(rec[period.segment == s]) for s in SEGMENTS},
    }
    return out, rec


def timed(fn, *args, **kw):
    t0 = time.perf_counter()
    out = fn(*args, **kw)
    return out, time.perf_counter() - t0


def rounded(x, n=3):
    """x rounded to n places (None or ... unchanged: the task is not done yet)."""
    return x if x is None or x is ... else round(float(x), n)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The situation

A UK online shop sells gifts and homeware. Many of its customers are small shops that buy to resell, and they order again and again. Grace, the shop's e-commerce manager, wants a block of ten products on two pages: the account page of a customer who logs in, and the home page for everyone else. Jonah, the shop's only developer, will build what you recommend. He asks: "Which method, for whom, and what does it cost to run?" Grace and Jonah are made up; the data is real.

Your job: compare four families of methods (popularity, co-occurrence, ALS and a two-stage ranker) on an honest time split, by customer segment, and say what the shop should show and for whom. The [lesson page](https://learning.nextia-ai.com/courses/recommendations/m10/recommend-for-an-online-retailer/#the-situation) tells the full story.

## 2. The data

Run the next cell. It reads the invoice lines and cleans them. Each line of the output counts one cleaning step.

> **Predict.** What share of the 1,067,371 lines has no customer ID: under 5%, about 10%, or more than 20%?

> **Check.** You should see `kept 794,183`, `return lines 17,587` and 5,852 customers who bought 4,621 products.

In [ ]:
raw = read("data/online_retail_ii.csv.gz")
lines, steps = clean(raw)
for step, n in steps.items():
    print(f"{step:16} {n:>9,}")
names = product_names(lines)
bought = lines[~lines.returned]
print(f"{bought.customer.nunique():,} customers bought {bought['product'].nunique():,} products, "
      f"{lines.date.min():%Y-%m-%d} to {lines.date.max():%Y-%m-%d}")

**What this tells us.** 235,151 lines have no customer ID: we cannot personalise them, so they leave. The lines that start with `C` are returns. We keep them apart: a product that a customer returned in full is not something to learn from.

## 3. The time split and the segments

Run the next cell. It makes the test period: November 2011. The methods learn from everything before 1 November, and we score them on what each customer bought in November. Each customer gets a segment from their invoices before November: **new** (0), **occasional** (1 or 2) or **loyal** (3 or more).

> **Check.** You should see 1,656 customers: 190 new, 356 occasional and 1,110 loyal.

In [ ]:
test = Period(lines, "2011-11-01", "2011-12-01")
words = [product_words(names.get(p, "")) for p in test.products]
print(f"{len(test.customers):,} customers bought in November 2011")
print(pd.Series(test.segment).value_counts().reindex(SEGMENTS).to_string())
print(f"learn from: {len(test.users):,} customers x {len(test.products):,} products; "
      f"{test.returned_pairs:,} pairs returned in full are left out")

These customers buy a lot: on average **33.2 different products** in the month (median 21). A list of 10 can find at most 10 of them, so recall@10 will look small. Compare methods with each other, not with Larkfield's numbers.

> **Your turn.** What share of the November (customer, product) pairs did the customer already buy before November? Use `test.truth` (customer → set of products bought in November) and `test.pairs` (one row per customer and product bought before, with columns `customer` and `product`). Store the share in `share_bought_before`.

In [ ]:
# Your answer: the share of November pairs that the customer bought before.
share_bought_before = ...

In [ ]:
expect('share_bought_before', rounded(share_bought_before), 0.39, tolerance=0.002)

**What this tells us.** A large part of what these customers buy, they bought before. Keep this number in mind: it decides the case study.

## 4. The baselines: popular and trending

Run the next cell. `popular` ranks products by the number of customers who ever bought them. `trending` does the same for the last 28 days. Everyone gets the same list.

In [ ]:
def popular(period):
    return np.argsort(-period.popularity(), kind="stable")


def trending(period, days=28):
    return np.argsort(-period.popularity(days), kind="stable")


results = {}
blank = np.full((len(test.customers), K), -1)
lists = {"popular": fill(blank, popular(test)), "trending": fill(blank, trending(test))}
for name in lists:
    results[name], _ = evaluate(test, lists[name], words)
    print(f"{name:10} recall@10 {results[name]['recall'][0]:.3f}")

> **Check.** You should see popular 0.035 and trending 0.047.

## 5. Buy again

The simplest personal list: the customer's own products, the ones bought on the most invoices first, then the most recent. A new customer has no products, so the list is filled with trending.

> **Predict.** Will "buy again" beat trending? By a little, or by a lot?

In [ ]:
def buy_again_scores(period, rows):
    """The customer's own products: more invoices first, then the most recent."""
    X, D = period.X[rows].toarray(), period.D[rows].toarray()
    recency = np.divide(1.0, D, out=np.zeros_like(D), where=D > 0)
    return np.where(X > 0, X + recency, 0.0)


lists["buy-again"] = personal(test, lambda rows: buy_again_scores(test, rows), trending(test))
results["buy-again"], _ = evaluate(test, lists["buy-again"], words)
print(f"buy-again  recall@10 {results['buy-again']['recall'][0]:.3f}")

> **Check.** You should see 0.139.

## 6. Neighbours: co-occurrence and item-item

Run the next cell. **Co-occurrence** counts the customers who bought both products. **Item-cf** divides that count by the size of each product (cosine), so bestsellers do not win every pair. Both score a product by summing its similarity to each product the customer bought. The lesson [Item-item collaborative filtering](https://learning.nextia-ai.com/courses/recommendations/m04/item-item-collaborative-filtering/) explains both.

In [ ]:
def cooc_matrix(period):
    """How many customers bought both products (the diagonal set to 0)."""
    C = (period.B.T @ period.B).toarray().astype(np.float32)
    np.fill_diagonal(C, 0)
    return C


def cosine_matrix(period):
    """Item-item cosine: shared customers / sqrt(customers of A x customers of B)."""
    C = cooc_matrix(period)
    n = np.sqrt(period.item_customers).astype(np.float32) + 1e-9
    return C / n[:, None] / n[None, :]


def neighbour_scores(period, S, rows):
    """Sum over the customer's products of (weight x similarity); weight = 1 + log(invoices)."""
    W = period.X[rows].copy()
    W.data = 1 + np.log(W.data)
    return np.asarray(W @ S)


C = cooc_matrix(test)
S = cosine_matrix(test)
lists["co-occurrence"] = personal(test, lambda rows: neighbour_scores(test, C, rows), trending(test))
lists["item-cf"] = personal(test, lambda rows: neighbour_scores(test, S, rows), trending(test))
for name in ("co-occurrence", "item-cf"):
    results[name], _ = evaluate(test, lists[name], words)
    print(f"{name:14} recall@10 {results[name]['recall'][0]:.3f}")

> **Check.** You should see co-occurrence 0.047 and item-cf 0.063.

## 7. ALS

Run the next cell. ALS learns a short vector for each customer and each product, as in [Hidden tastes](https://learning.nextia-ai.com/courses/recommendations/m05/hidden-tastes/). We use the settings that were best on the **validation** period (October 2011): 256 factors and regularisation 10. The grid of 12 settings is on the lesson page; it is not repeated here, to keep the notebook fast. The fit takes about 5 seconds.

In [ ]:
def fit_als(period, factors=64, regularization=0.1, alpha=1.0, iterations=15, seed=37):
    import implicit
    from threadpoolctl import threadpool_limits
    conf = period.X.copy().astype(np.float32)
    conf.data = alpha * (1 + np.log(conf.data))
    with threadpool_limits(1):
        model = implicit.cpu.als.AlternatingLeastSquares(factors=factors, regularization=regularization,
                                                         iterations=iterations, random_state=seed, num_threads=1)
        model.fit(conf, show_progress=False)
    return model


def als_scores(model, rows):
    return np.asarray(model.user_factors[rows] @ model.item_factors.T)


ALS = {"factors": 256, "regularization": 10.0}
als = fit_als(test, **ALS)
lists["als"] = personal(test, lambda rows: als_scores(als, rows), trending(test))
results["als"], _ = evaluate(test, lists["als"], words)
print(f"als  recall@10 {results['als']['recall'][0]:.3f}")

> **Check.** You should see 0.122.

## 8. The two-stage ranker

Run the next cell. It defines the ranker of [Why two stages](https://learning.nextia-ai.com/courses/recommendations/m07/why-two-stages/) and [A learned ranker](https://learning.nextia-ai.com/courses/recommendations/m07/a-learned-ranker/), changed for this shop. Four sources make the candidates: trending (100), item-cf (100), ALS (100) and the customer's own products (50). LightGBM learns to order them from 18 features. It prints nothing.

In [ ]:
# The two-stage ranker: candidate sources, features, training on the month before, ranking.
SOURCES = {"trending": 100, "item_cf": 100, "als": 100, "own": 50}


FEATURES = ["trending_rank", "item_cf_score", "item_cf_rank", "als_score", "als_rank", "own_rank", "n_sources",
            "pop_all", "pop_28", "pop_7", "last_year", "log_price", "product_age", "user_invoices",
            "user_days_since_last", "user_products", "pair_invoices", "pair_days_since"]


class Stage:
    """Candidate sources and features for one cut-off."""

    def __init__(self, period, als_settings):
        self.p = p = period
        self.S = cosine_matrix(p)
        self.als = fit_als(p, **als_settings)
        self.trend = trending(p)
        self.pop_all = np.log1p(p.popularity())
        self.pop_28 = np.log1p(p.popularity(28))
        self.pop_7 = np.log1p(p.popularity(7))
        a = p.start - pd.Timedelta(days=365)
        ly = p.buys[(p.buys["date"] >= a) & (p.buys["date"] < a + (p.end - p.start))]
        self.last_year = np.log1p(ly.groupby("product")["customer"].nunique().reindex(p.products).fillna(0).to_numpy())
        price = p.buys.groupby("product")["price"].median().reindex(p.products).to_numpy()
        self.log_price = np.log(price)
        first = p.buys.groupby("product")["date"].min().reindex(p.products)
        self.age = np.minimum((p.start - first).dt.days.to_numpy(), 999)
        self.n_products = np.asarray(p.B.sum(1)).ravel()

    def candidates(self, customers):
        p = self.p
        rows = np.array([p.uidx.get(c, -1) for c in customers])
        parts = []
        t = self.trend[:SOURCES["trending"]]
        parts.append(pd.DataFrame({"c": np.repeat(np.arange(len(customers)), len(t)), "p": np.tile(t, len(customers)),
                                   "trending_rank": np.tile(np.arange(1, len(t) + 1), len(customers))}))
        known = np.flatnonzero(rows >= 0)
        if len(known):
            r = rows[known]
            for name, scores in (("item_cf", neighbour_scores(p, self.S, r)), ("als", als_scores(self.als, r)),
                                 ("own", buy_again_scores(p, r))):
                n = SOURCES[name]
                top = top_k(scores, n)
                sc = np.take_along_axis(scores, top, axis=1)
                ok = sc > 0 if name != "als" else np.ones_like(sc, bool)
                rank = np.tile(np.arange(1, n + 1), (len(r), 1))
                cols = {"c": np.repeat(known, n)[ok.ravel()], "p": top[ok], f"{name}_rank": rank[ok]}
                if name != "own":
                    cols[f"{name}_score"] = sc[ok]
                parts.append(pd.DataFrame(cols))
        cand = pd.concat(parts, ignore_index=True).groupby(["c", "p"], sort=True).first().reset_index()
        return cand, rows

    def features(self, cand, rows, customers):
        p = self.p
        f = cand.copy()
        for name in SOURCES:
            col = f"{name}_rank"
            f[col] = f[col].fillna(999) if col in f else 999.0
        for name in ("item_cf", "als"):
            col = f"{name}_score"
            f[col] = f[col].fillna(0.0) if col in f else 0.0
        f["n_sources"] = sum((f[f"{n}_rank"] < 999).astype(int) for n in SOURCES)
        q = f["p"].to_numpy()
        f["pop_all"], f["pop_28"], f["pop_7"] = self.pop_all[q], self.pop_28[q], self.pop_7[q]
        f["last_year"], f["log_price"], f["product_age"] = self.last_year[q], self.log_price[q], self.age[q]
        r = rows[f["c"].to_numpy()]
        cust = np.asarray(customers)[f["c"].to_numpy()]
        f["user_invoices"] = p.invoices_before.reindex(cust).fillna(0).to_numpy()
        last = p.last_day.reindex(cust)
        f["user_days_since_last"] = np.minimum((p.start - last).dt.days.fillna(999).to_numpy(), 999)
        f["user_products"] = np.where(r >= 0, self.n_products[np.maximum(r, 0)], 0)
        known = r >= 0
        inv = np.zeros(len(f))
        since = np.full(len(f), 999.0)
        if known.any():
            inv[known] = np.asarray(p.X[r[known], q[known]]).ravel()
            d = np.asarray(p.D[r[known], q[known]]).ravel()
            since[known] = np.where(d > 0, d - 1, 999)
        f["pair_invoices"], f["pair_days_since"] = inv, since
        return f


def fit_ranker(lines, train_start, train_end, als_settings, negatives=100, trees=300, seed=37):
    """Learn the ranker on one window BEFORE the period: candidates and features from before `train_start`, labels =
    what customers bought from `train_start` to the day before `train_end`."""
    import lightgbm as lgb
    p = Period(lines, train_start, train_end)
    stage = Stage(p, als_settings)
    customers = p.customers
    cand, rows = stage.candidates(customers)
    bought = [{p.pidx[x] for x in p.truth[c] if x in p.pidx} for c in customers]
    cand["label"] = [int(q in bought[c]) for c, q in zip(cand["c"].to_numpy(), cand["p"].to_numpy())]
    rng = np.random.default_rng(seed)
    cand["r"] = rng.random(len(cand))
    cand["nr"] = cand[cand.label == 0].groupby("c")["r"].rank(method="first")
    with_pos = cand.loc[cand.label == 1, "c"].unique()
    cand = cand[((cand.label == 1) | (cand.nr <= negatives)) & cand["c"].isin(with_pos)].sort_values("c")
    f = stage.features(cand.drop(columns=["label", "r", "nr"]), rows, customers)
    params = {"objective": "lambdarank", "learning_rate": 0.05, "num_leaves": 31, "min_data_in_leaf": 20,
              "bagging_fraction": 0.8, "bagging_freq": 1, "feature_fraction": 0.8, "seed": seed, "num_threads": 1,
              "verbose": -1, "deterministic": True, "force_row_wise": True}
    group = f.groupby("c", sort=True).size().to_numpy()
    model = lgb.train(params, lgb.Dataset(f[FEATURES], label=cand["label"].to_numpy(), group=group),
                      num_boost_round=trees)
    info = {"customers": int(len(group)), "rows": int(len(f)), "positives": int(cand["label"].sum())}
    return model, info


def two_stage(period, model, als_settings, k=K, drop_bought=False):
    """Lists of the ranker: candidates and features from before the period, best score first."""
    stage = Stage(period, als_settings)
    cand, rows = stage.candidates(period.customers)
    f = stage.features(cand, rows, period.customers)
    f["score"] = model.predict(f[FEATURES])
    if drop_bought:
        f = f[f["pair_invoices"] == 0]
    f = f.sort_values(["c", "score"], ascending=[True, False])
    f["place"] = f.groupby("c").cumcount()
    f = f[f["place"] < k]
    lists = np.full((len(period.customers), k), -1)
    lists[f["c"].to_numpy(), f["place"].to_numpy()] = f["p"].to_numpy()
    return fill(lists, stage.trend, k), stage, cand

Run the next cell. The ranker learns on **October**: candidates and features from the data before 1 October, and labels from what customers bought in October. Then it ranks the candidates for November, with features from the data before 1 November. It never sees November. This takes about a minute.

> **Check.** You should see about 1,305 training customers and recall@10 0.159.

In [ ]:
start = time.perf_counter()
ranker, info = fit_ranker(lines, "2011-10-01", "2011-11-01", ALS)
lists["two-stage"], stage, candidates = two_stage(test, ranker, ALS)
results["two-stage"], _ = evaluate(test, lists["two-stage"], words)
print(info)
print(f"two-stage  recall@10 {results['two-stage']['recall'][0]:.3f}  ({time.perf_counter() - start:.0f} s)")

## 9. Compare, by segment

Run the next cell. It shows every method on the same customers, with the same measures. The last three columns are recall@10 by segment.

In [ ]:
def table(results):
    """One row per method: recall@10 (95% interval), NDCG@10, coverage and recall by segment."""
    rows = []
    for name, o in results.items():
        rows.append({"method": name, "recall@10": round(o["recall"][0], 3),
                     "interval": f"{o['recall'][1]:.3f} to {o['recall'][2]:.3f}", "NDCG@10": round(o["ndcg"], 3),
                     "new to the customer": round(o["discovery"], 3), "coverage": f"{o['coverage']:.1%}",
                     **{s: round(o["segments"][s][0], 3) for s in SEGMENTS}})
    return pd.DataFrame(rows).set_index("method")


table(results)

Two intervals that overlap do not prove that two methods are the same, and two that do not overlap hide how many customers changed. Compare the methods on the **same customers**, as in [Evaluate by segment](https://learning.nextia-ai.com/courses/recommendations/m03/evaluate-by-segment/).

> **Your turn.** `evaluate` also returns recall@10 for each customer (its second value). Compute it for `lists["two-stage"]` and `lists["buy-again"]`. Store in `better` the number of customers for whom the two-stage list has the higher recall@10.

In [ ]:
# Your answer: for how many customers is two-stage better than buy-again?
better = ...

In [ ]:
expect('better', better, 672)

**What this tells us.** The ranker is better for 672 customers, worse for 299 and the same for 685. The mean difference is small (+0.020), and its 95% interval on the lesson page does not include zero.

**What this tells us.** For known customers, the two-stage ranker is first and buy-again second; the neighbours and the popularity lists are far behind. For **new** customers, every personal method falls back to trending, and the ranker scores lower than trending (0.035 against 0.044; with only 190 new customers, the difference is not clear).

> **Your turn.** Make the design that the lesson recommends: the two-stage lists for known customers, and the trending list for new customers (`test.segment == "new"`). Store the lists in `design` (a copy of `lists["two-stage"]` with the new customers' rows changed) and its recall@10 in `design_recall`.

In [ ]:
# Your answer: two-stage for known customers, trending for new customers.
design = ...
design_recall = ...

In [ ]:
expect('design_recall', rounded(design_recall), 0.16, tolerance=0.001)

## 10. Failure case: a rule from another shop

Larkfield's rules remove products that the customer already bought, because most of its products are bought once. This shop is different.

> **Predict.** If we remove every product the customer bought before, what happens to recall@10 and to "new to the customer"?

Run the next cell. `drop_bought=True` removes the customer's own products before the top 10 are taken.

In [ ]:
rule_lists, _, _ = two_stage(test, ranker, ALS, drop_bought=True)
rule, _ = evaluate(test, rule_lists, words)
print(f"two-stage           recall@10 {results['two-stage']['recall'][0]:.3f}  new to the customer {results['two-stage']['discovery']:.3f}")
print(f"with the Larkfield rule  recall@10 {rule['recall'][0]:.3f}  new to the customer {rule['discovery']:.3f}")

> **Check.** You should see recall@10 fall from 0.159 to 0.030, and "new to the customer" rise from 0.008 to 0.059.

**What this tells us.** A rule is a business decision about one shop. Here, reminding a reseller of what they buy every month is most of the value. Finding new products is a different job, for a different block of the page.

## 11. End to end on new input: 1 to 9 December 2011

The last nine days of the data were never used. Run the next cell. It does the whole pipeline again, one month later: the ranker learns on November, then ranks for December with everything known before 1 December. New customers get trending.

> **Check.** You should see 610 customers and recall@10 0.207 for the design (trending 0.052, buy-again 0.173).

In [ ]:
december = Period(lines, "2011-12-01", "2011-12-10")
words_dec = [product_words(names.get(p, "")) for p in december.products]
ranker_dec, _ = fit_ranker(lines, "2011-11-01", "2011-12-01", ALS)
two, stage_dec, _ = two_stage(december, ranker_dec, ALS)
new = december.segment == "new"
final = two.copy()
final[new] = fill(np.full((new.sum(), K), -1), trending(december))
check = {"trending": fill(np.full((len(december.customers), K), -1), trending(december)),
         "buy-again": personal(december, lambda rows: buy_again_scores(december, rows), trending(december)),
         "two-stage + trending for new": final}
print(f"{len(december.customers):,} customers bought from 1 to 9 December")
table({name: evaluate(december, l, words_dec)[0] for name, l in check.items()})

Run the next cell. It prints the list of one loyal customer, with a mark on each product the customer bought from 1 to 9 December, and the number of invoices with that product before December.

In [ ]:
customer = '12347'
i = list(december.customers).index(customer)
row = december.uidx[customer]
bought = december.truth[customer]
for place, p in enumerate(final[i], 1):
    code = december.products[p]
    print(f"{place:2}. {'bought' if code in bought else '      '}  {int(december.X[row, p]):2} invoices before  {names.get(code, '')[:40]}")

> **Check.** The first lines are products this customer buys often. You should see 3 products marked `bought`.

## 12. Recap

You cleaned real invoice lines (repeats, missing customer IDs, non-products, returns), split them by time, and compared popularity, buy-again, co-occurrence, item-cf, ALS and a two-stage ranker on the same customers, by segment. On this shop, the customer's own history is the strongest signal, the ranker adds to it, and new customers are best served by trending. The [lesson page](https://learning.nextia-ai.com/courses/recommendations/m10/recommend-for-an-online-retailer/#the-recommendation) turns this into a recommendation with its costs, its limits and what a person still checks.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Next item from browsing sessions](https://learning.nextia-ai.com/courses/recommendations/m10/next-item-from-browsing-sessions/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/recommendations/m10/recommend-for-an-online-retailer/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The data is real: UCI Online Retail II by Daqing Chen, CC BY 4.0. Customer IDs are numbers only.